# Femicide Graph Pipeline v7-2-3

Heterogeneous property graph for a single IPV/femicide case from STAT_FW.
Implements design principles P1–P9 (v7 schema).

## Design Principles

| # | Name | Rule |
|---|------|------|
| P1 | No incident nodes | Incidents removed; replaced by direct dyadic edges. |
| P2 | Dyadic primacy | Interactions are direct offender↔victim edges. No incident mediation. |
| P3 | Mediation for artefacts | Physical objects (weapon, method, location) sit on the dyadic path (descriptive only). |
| P4 | No target leakage | No KILLED edge, no outcome-stage content in predictive variant. |
| P5 | Two explicit graph variants | DESCRIPTIVE (full case) and PREDICTIVE (prior-stage only). |
| P6 | Consistent three-state logic | Every binary: Present (value=1, observed=1), Absent (value=0, observed=1), Unknown (observed=0). |
| P7 | Explicit boundary criterion | Dyadic edge = requires both persons. Otherwise → attribute or node feature. |
| P8 | Deterministic, idempotent | Stable IDs, same input → same output. |
| P9 | No star topologies | Risk factors as node attributes (rf_* prefix), not leaf nodes. |

In [ ]:
import re
import json
import hashlib
import datetime as dt
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import networkx as nx

DATA_PATH     = Path("Stat_FW.xlsx")
SHEET_IDX     = 0
CASE_ID       = "4200-73111-00001-22"   # Case 1 (design case)
# CASE_ID     = "3300-73111-00001-20"   # Case 2 (validation case)

GRAPH_VARIANT = "predictive"   # "descriptive" or "predictive" (P5)
assert GRAPH_VARIANT in {"descriptive", "predictive"}, f"Invalid variant: {GRAPH_VARIANT}"

EXPORT_DIR = Path.cwd() / f"exports_v7_{GRAPH_VARIANT}"
EXPORT_DIR.mkdir(exist_ok=True)

PLACEHOLDERS = {"not applicable", "none", "unknown", "not known", "n/a", "", "nan"}

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Missing data file: {DATA_PATH}")

df = pd.read_excel(DATA_PATH, sheet_name=SHEET_IDX)
subset = df[df["case_id"] == CASE_ID]
if subset.empty:
    raise ValueError(f"case_id not found: {CASE_ID}")

row = subset.iloc[0]
cid = str(row["case_id"])

case_id     = f"CASE_{cid}"
offender_id = f"PERSON_{cid}_O"
victim_id   = f"PERSON_{cid}_V"

print(f"Config: variant={GRAPH_VARIANT}, case={cid}")
print(f"Export dir: {EXPORT_DIR}")

In [ ]:

EXPECTED_COLUMNS = [
    # Case admin
    "case_id", "crime_date", "verdict_date", "Crime_verdict_timegap",
    "crime_arrest_timegap", "crime_report_date", "court_name", "court_number",
    "internal_police_number", "case_solved", "case_type", "region",
    "type_of_femicide", "children_present",
    # Dyadic state
    "separated", "length_of_separation_in_months", "victim_new_partner",
    "child_custody_access_disputes", "shared_children", "shared_children_number",
    "prior_family_court_involvement",
    # Person attributes — offender
    "offender_age", "offender_gender", "offender_mental_health",
    "offender_nationality", "offender_employment", "offender_education",
    "offender_marital_status", "offender_disability",
    # Person attributes — victim
    "victim_age", "victim_gender", "victim_mental_health",
    "victim_nationality", "victim_employment", "victim_education",
    "victim_residency_status", "victim_race_ethnicity",
    "victim_marital_status", "victim_disability",
    # Children
    "victim_children", "number_of_victim_children",
    "offender_children", "number_of_offender_children",
    "shared_children", "children_witness", "children_witness_how",
    "offender_threatened_harmed_children",
    # Dyadic prior actions
    "prior_violence", "type_of_prior_violence",
    "control", "prior_attempts_isolate_victim",
    "prior_strangulation", "prior_threats_with_weapon",
    "prior_assault_with_weapon",
    # Dyadic behaviours
    "escalation_of_violence", "obsessive_behaviour",
    "sexual_jealousy", "misogynistic_attitudes",
    "controlled_victims_daily_activities", "threats_of_suicide",
    "youth_couple",
    # Descriptive-only
    "sexual_violence_part_of_crime",
    "weapon_type", "method_of_killing", "location_of_crime",
    "motive",
    # Risk factors — offender
    "offender_history_violence_outside_family",
    "offender_history_domestic_violence_current",
    "offender_history_domestic_violence_past",
    "prior_threats_to_kill_other",
    "prior_suicide_attempt",
    "prior_suicide_threats",
    "prior_sexual_assault_others",
    "excessive_alcohol_drug_use",
    "offender_depressed_family_opinion",
    "offender_depressed_professional",
    "access_or_possession_firearms",
    "offender_suicide",
    "offender_suicide_attempt",
    "offender_access_to_victim_after_assessment",
    "prior_hostage_taking",
    "prior_destruction_of_property",
    "prior_violence_against_pets",
    "prior_assault_while_pregnant",
    # Risk factors — victim
    "victim_considered_vulnerable",
    "victim_pregnant",
    "victim_disability",
    "victim_reported_to_authorities",
    "victim_womens_shelter",
    "victim_risk_assessment_made",
    "victim_criminal_history",
    "victim_history_abuse_as_victim",
    "victim_history_abuse_as_offender",
    # Additional offender risk
    "offender_criminal_history",
    "offender_history_abuse_as_victim",
]

missing = [c for c in EXPECTED_COLUMNS if c not in df.columns]
if missing:
    print(f"⚠ Missing {len(missing)} expected column(s): {missing}")
else:
    print(f"✓ All {len(EXPECTED_COLUMNS)} expected columns present")
print(f"  DataFrame shape: {df.shape}")

In [ ]:
# ── Cell 3: Graph state + helper functions ──────────────────────────

nodes = {}   # node_id → dict of attrs
edges = {}   # edge_id → dict of attrs


def norm_str(x):
    """Normalise raw cell value. Returns None for blanks/placeholders."""
    if x is None:
        return None
    if isinstance(x, float) and np.isnan(x):
        return None
    s = str(x).strip()
    if s.lower() in PLACEHOLDERS:
        return None
    return s


def parse_bool(x):
    """True / False / None three-state parse."""
    s = norm_str(x)
    if s is None:
        return None
    sl = s.lower()
    if sl in {"yes", "y", "true", "1"} or sl.startswith("yes"):
        return True
    if sl in {"no", "n", "false", "0"} or sl.startswith("no"):
        return False
    return None


def parse_stage_text(x):
    """Extract stage from raw text. Returns prior/during/post/None."""
    s = norm_str(x)
    if s is None:
        return None
    sl = s.lower()
    if "before" in sl or "prior" in sl:
        return "prior"
    if "during" in sl:
        return "during"
    if "after" in sl or "post" in sl:
        return "post"
    return None


def stable_edge_id(source_id, target_id, edge_type, edge_key):
    """Deterministic edge ID — SHA1 of key components (P8)."""
    raw = f"{source_id}||{target_id}||{edge_type}||{edge_key}"
    return "E_" + hashlib.sha1(raw.encode()).hexdigest()[:16]


def add_attr(d, k, v):
    """Set d[k] = v only if norm_str(v) is not None."""
    v2 = norm_str(v)
    if v2 is not None:
        d[k] = v2


def add_node(node_id, node_type, **attrs):
    """Get-or-create node. Merges attrs idempotently."""
    n = nodes.get(node_id, {"node_id": node_id, "node_type": node_type})
    for k, v in attrs.items():
        add_attr(n, k, v)
    nodes[node_id] = n
    return node_id


def merge_edge(source_id, target_id, edge_type, edge_key=None, **attrs):
    """Get-or-create edge. Merges attrs idempotently (P8)."""
    if edge_key is None:
        edge_key = edge_type
    eid = stable_edge_id(source_id, target_id, edge_type, edge_key)
    e = edges.get(eid, {
        "edge_id": eid,
        "source_id": source_id,
        "target_id": target_id,
        "edge_type": edge_type,
        "edge_key": edge_key,
    })
    for k, v in attrs.items():
        add_attr(e, k, v)
    edges[eid] = e
    return eid


def merge_edge_threestate(source_id, target_id, edge_type, bool_value,
                          edge_key=None, **attrs):
    """Three-state edge creation (P6).
    True  → value=1, observed=1
    False → value=0, observed=1
    None  →          observed=0
    """
    if bool_value is True:
        attrs["value"] = 1
        attrs["observed"] = 1
    elif bool_value is False:
        attrs["value"] = 0
        attrs["observed"] = 1
    else:
        attrs["observed"] = 0
    return merge_edge(source_id, target_id, edge_type, edge_key=edge_key, **attrs)


def set_factor_attrs(node_id, factor_key, bool_value, stage):
    """Three-state factor as node attributes (P9).
    Adds rf_{factor_key}_value, rf_{factor_key}_observed, rf_{factor_key}_stage
    directly to the node dict.
    Always sets _value to ensure consistent feature-vector dimensions
    across cases (Section 6.5 requirement).
    """
    n = nodes[node_id]
    if bool_value is True:
        n[f"rf_{factor_key}_value"] = 1
        n[f"rf_{factor_key}_observed"] = 1
    elif bool_value is False:
        n[f"rf_{factor_key}_value"] = 0
        n[f"rf_{factor_key}_observed"] = 1
    else:
        n[f"rf_{factor_key}_value"] = 0.5   # unknown → midpoint (ensures uniform vec dim)
        n[f"rf_{factor_key}_observed"] = 0
    n[f"rf_{factor_key}_stage"] = stage


def slugify(s):
    """Convert string to lowercase slug for stable node IDs."""
    v = norm_str(s)
    if v is None:
        return None
    return re.sub(r"[^a-z0-9]+", "_", v.lower()).strip("_")


print("✓ Helpers loaded.")

In [ ]:
# ── Cell 4: Core nodes & dyadic state (P2, P6) ─────────────────────

# ── Case node ──
add_node(case_id, "case", label=cid)
for k in ["crime_date", "verdict_date", "Crime_verdict_timegap",
          "crime_arrest_timegap", "case_solved", "case_type",
          "court_number", "internal_police_number",
          "type_of_femicide", "children_present"]:
    add_attr(nodes[case_id], k, row.get(k))

# ── Person nodes ──
add_node(offender_id, "person", label="Offender", role="offender",
         age=row.get("offender_age"),
         gender=row.get("offender_gender"),
         mental_health=row.get("offender_mental_health"),
         nationality=row.get("offender_nationality"),
         employment=row.get("offender_employment"),
         education=row.get("offender_education"),
         marital_status=row.get("offender_marital_status"),
         disability=row.get("offender_disability"))

add_node(victim_id, "person", label="Victim", role="victim",
         age=row.get("victim_age"),
         gender=row.get("victim_gender"),
         mental_health=row.get("victim_mental_health"),
         nationality=row.get("victim_nationality"),
         employment=row.get("victim_employment"),
         education=row.get("victim_education"),
         residency_status=row.get("victim_residency_status"),
         race_ethnicity=row.get("victim_race_ethnicity"),
         marital_status=row.get("victim_marital_status"),
         disability=row.get("victim_disability"))

# ── Structural: case → participants ──
merge_edge(case_id, offender_id, "HAS_PARTICIPANT", role="offender", scope="case")
merge_edge(case_id, victim_id,   "HAS_PARTICIPANT", role="victim",   scope="case")

# ── Dyadic relational state (P2, P6 — always created with three-state) ──

# SEPARATED_FROM: victim → offender
sep_bool = parse_bool(row.get("separated"))
sep_attrs = {"scope": "case", "stage": "prior"}
months = norm_str(row.get("length_of_separation_in_months"))
if months is not None:
    sep_attrs["months"] = months
merge_edge_threestate(victim_id, offender_id, "SEPARATED_FROM", sep_bool, **sep_attrs)

# HAS_NEW_PARTNER: victim → offender
merge_edge_threestate(victim_id, offender_id, "HAS_NEW_PARTNER",
                      parse_bool(row.get("victim_new_partner")),
                      scope="case", stage="prior")

# CHILD_CUSTODY_DISPUTE: offender → victim (unidirectional; ToUndirected in GNN)
custody_bool = parse_bool(row.get("child_custody_access_disputes"))
merge_edge_threestate(offender_id, victim_id, "CHILD_CUSTODY_DISPUTE", custody_bool,
                      scope="case", stage="prior")

# SHARED_CHILDREN: offender → victim
shared_bool = parse_bool(row.get("shared_children"))
sc_attrs = {"scope": "case", "stage": "prior"}
shared_n = norm_str(row.get("shared_children_number"))
if shared_n is not None:
    sc_attrs["n"] = shared_n
merge_edge_threestate(offender_id, victim_id, "SHARED_CHILDREN", shared_bool, **sc_attrs)

# PRIOR_FAMILY_COURT_INVOLVEMENT: offender → victim (v7-2-3 addition)
merge_edge_threestate(offender_id, victim_id, "PRIOR_FAMILY_COURT_INVOLVEMENT",
                      parse_bool(row.get("prior_family_court_involvement")),
                      scope="case", stage="prior")

print(f"Core nodes: {len(nodes)}, edges: {len(edges)}")

In [ ]:
# ── Cell 5: Children nodes ──────────────────────────────────────────

child_nodes = []
child_counter = 1

def _int_from_maybe(x):
    x = norm_str(x)
    if x is None:
        return None
    try:
        return int(str(x).split()[0])
    except:
        return None

victim_n   = _int_from_maybe(row.get("number_of_victim_children"))   if parse_bool(row.get("victim_children"))   is True else 0
offender_n = _int_from_maybe(row.get("number_of_offender_children")) if parse_bool(row.get("offender_children")) is True else 0
shared_n_c = _int_from_maybe(row.get("shared_children_number"))      if parse_bool(row.get("shared_children"))   is True else 0
victim_n   = victim_n   or 0
offender_n = offender_n or 0
shared_n_c = shared_n_c or 0

# Shared children
for _ in range(shared_n_c):
    ch_id = f"PERSON_{cid}_CHILD_{child_counter}"
    add_node(ch_id, "person", label=f"Child {child_counter} (shared)", role="child")
    merge_edge(case_id,     ch_id, "HAS_PARTICIPANT", role="child", scope="case")
    merge_edge(victim_id,   ch_id, "PARENT_OF", scope="case")
    merge_edge(offender_id, ch_id, "PARENT_OF", scope="case")
    child_nodes.append(ch_id)
    child_counter += 1

# Victim-only children
for _ in range(max(victim_n - shared_n_c, 0)):
    ch_id = f"PERSON_{cid}_CHILD_{child_counter}"
    add_node(ch_id, "person", label=f"Child {child_counter} (victim's)", role="child")
    merge_edge(case_id,   ch_id, "HAS_PARTICIPANT", role="child", scope="case")
    merge_edge(victim_id, ch_id, "PARENT_OF", scope="case")
    child_nodes.append(ch_id)
    child_counter += 1

# Offender-only children
for _ in range(max(offender_n - shared_n_c, 0)):
    ch_id = f"PERSON_{cid}_CHILD_{child_counter}"
    add_node(ch_id, "person", label=f"Child {child_counter} (offender's)", role="child")
    merge_edge(case_id,     ch_id, "HAS_PARTICIPANT", role="child", scope="case")
    merge_edge(offender_id, ch_id, "PARENT_OF", scope="case")
    child_nodes.append(ch_id)
    child_counter += 1

children_present = parse_bool(row.get("children_present"))
children_witness = parse_bool(row.get("children_witness"))
children_witness_how = norm_str(row.get("children_witness_how"))

# WITNESSED_CRIME / PRESENT_AT_CRIME — descriptive only (during-stage)
if GRAPH_VARIANT == "descriptive" and child_nodes:
    for ch_id in child_nodes:
        merge_edge_threestate(
            ch_id, case_id, "WITNESSED_CRIME", children_witness,
            scope="case", stage="during",
        )
        if children_witness_how:
            nodes[ch_id]["witnessed_how"] = children_witness_how

print(f"Created {len(child_nodes)} child node(s). Total nodes: {len(nodes)}, edges: {len(edges)}")

In [ ]:
# ── Cell 6: Dyadic prior actions (P1, P2, P6) ──────────────────────
# Each tuple: (column_name, edge_type, stage)

PRIOR_ACTION_SPECS = [
    ("prior_violence",               "PRIOR_VIOLENCE",        "prior"),
    ("control",                      "COERCIVE_CONTROL",      "prior"),
    ("prior_attempts_isolate_victim", "ATTEMPTED_ISOLATION",   "prior"),
    ("prior_strangulation",          "PRIOR_STRANGULATION",   "prior"),
    ("prior_threats_with_weapon",    "THREATENED_WITH_WEAPON","prior"),
    ("prior_assault_with_weapon",    "ASSAULTED_WITH_WEAPON", "prior"),
]

for col, edge_type, stage in PRIOR_ACTION_SPECS:
    b = parse_bool(row.get(col))
    extra = {}
    if col == "prior_violence":
        detail = norm_str(row.get("type_of_prior_violence"))
        if detail:
            extra["detail"] = detail
    merge_edge_threestate(
        offender_id, victim_id, edge_type, b,
        scope="case", stage=stage, source_col=col, **extra,
    )

# SEXUAL_VIOLENCE — descriptive only (during-stage)
if GRAPH_VARIANT == "descriptive":
    merge_edge_threestate(
        offender_id, victim_id, "SEXUAL_VIOLENCE",
        parse_bool(row.get("sexual_violence_part_of_crime")),
        scope="case", stage="during",
    )

print(f"Prior actions done. Edges: {len(edges)}")

In [ ]:
# ── Cell 7: Dyadic behaviours + lookup tables ───────────────────────

# Columns → edge types (dyadic, offender → victim)
DYADIC_BEHAVIOR_MAP = {
    "escalation_of_violence":              "ESCALATED_VIOLENCE",
    "obsessive_behaviour":                 "STALKED",
    "sexual_jealousy":                     "SEXUAL_JEALOUSY",
    "misogynistic_attitudes":              "MISOGYNISTIC_ATTITUDES",
    "controlled_victims_daily_activities": "CONTROLLED_DAILY_ACTIVITIES",
    "threats_of_suicide":                  "THREATENED_SUICIDE",  # P7: coercive control
}

# Symmetric (bidirectional)
DYADIC_SYMMETRIC = {
    "youth_couple": "YOUTH_COUPLE",
}

# Canonical factor keys — disambiguate similarly-named columns
FACTOR_CANON = {
    "prior_suicide_attempt":    "suicide_attempt_prior",
    "offender_suicide_attempt": "suicide_attempt_during",
    "prior_suicide_threats":    "suicide_threats_prior",
    "threats_of_suicide":       "suicide_threats_during",
    "offender_suicide":         "suicide_post_incident",
}

# Stage overrides for specific columns
FACTOR_STAGE_DEFAULT = {
    "offender_suicide":         "post",
    # Note: offender_suicide_attempt was removed from FACTOR_STAGE_DEFAULT
    # because stage filtering in the predictive variant handles it via infer_stage
}


def canonical_factor_key(col_name):
    col_name = str(col_name).strip()
    key = FACTOR_CANON.get(col_name, col_name).lower()
    key = re.sub(r"^(offender|victim)_", "", key)
    key = re.sub(r"^prior_", "", key)
    return key


def infer_stage(raw_value, col):
    """Infer stage: parse text → FACTOR_STAGE_DEFAULT → heuristic → prior."""
    st = parse_stage_text(raw_value)
    if st:
        return st
    if col in FACTOR_STAGE_DEFAULT:
        return FACTOR_STAGE_DEFAULT[col]
    if col.startswith("prior_") or "history" in col:
        return "prior"
    if "suicide_attempt" in col and col.startswith("offender_"):
        return "during"
    return "prior"


# ── Create dyadic behaviour edges (always, three-state) ──
for col, edge_type in DYADIC_BEHAVIOR_MAP.items():
    b = parse_bool(row.get(col))
    merge_edge_threestate(
        offender_id, victim_id, edge_type, b,
        scope="case", stage="prior", source_col=col,
    )

# ── Symmetric behaviours (bidirectional) ──
for col, edge_type in DYADIC_SYMMETRIC.items():
    b = parse_bool(row.get(col))
    merge_edge_threestate(offender_id, victim_id, edge_type, b,
                          scope="case", stage="prior", source_col=col)
    merge_edge_threestate(victim_id, offender_id, edge_type, b,
                          scope="case", stage="prior", source_col=col)

print(f"Dyadic behaviours done. Edges: {len(edges)}")

In [ ]:
# ── Cell 8: Risk factors as node attributes (P6, P9) ────────────────

FACTOR_AS_ATTR_COLS = [
    "offender_history_violence_outside_family",
    "offender_history_domestic_violence_current",
    "offender_history_domestic_violence_past",
    "prior_threats_to_kill_other",
    "prior_suicide_attempt",
    "prior_suicide_threats",
    "prior_sexual_assault_others",
    "excessive_alcohol_drug_use",
    "offender_depressed_family_opinion",
    "offender_depressed_professional",
    "access_or_possession_firearms",
    "offender_suicide",
    "offender_suicide_attempt",
    "offender_access_to_victim_after_assessment",
    "prior_hostage_taking",
    "prior_destruction_of_property",
    "prior_violence_against_pets",
    "prior_assault_while_pregnant",
    "offender_criminal_history",
    "offender_history_abuse_as_victim",
    # Victim-side factors
    "victim_considered_vulnerable",
    "victim_pregnant",
    "victim_disability",
    "victim_reported_to_authorities",
    "victim_womens_shelter",
    "victim_risk_assessment_made",
    "victim_criminal_history",
    "victim_history_abuse_as_victim",
    "victim_history_abuse_as_offender",
]

# Explicit holder mapping (P8) — every factor has a named holder
RISK_HOLDER_MAP = {
    "offender_history_violence_outside_family":    "offender",
    "offender_history_domestic_violence_current":  "offender",
    "offender_history_domestic_violence_past":     "offender",
    "prior_threats_to_kill_other":                 "offender",
    "prior_suicide_attempt":                       "offender",
    "prior_suicide_threats":                       "offender",
    "prior_sexual_assault_others":                 "offender",
    "excessive_alcohol_drug_use":                  "offender",
    "offender_depressed_family_opinion":            "offender",
    "offender_depressed_professional":              "offender",
    "access_or_possession_firearms":               "offender",
    "offender_suicide":                            "offender",
    "offender_suicide_attempt":                    "offender",
    "offender_access_to_victim_after_assessment":  "offender",
    "prior_hostage_taking":                        "offender",
    "prior_destruction_of_property":               "offender",
    "prior_violence_against_pets":                 "offender",
    "prior_assault_while_pregnant":                "offender",
    "offender_criminal_history":                   "offender",
    "offender_history_abuse_as_victim":            "offender",
    "victim_considered_vulnerable":                "victim",
    "victim_pregnant":                             "victim",
    "victim_disability":                           "victim",
    "victim_reported_to_authorities":              "victim",
    "victim_womens_shelter":                       "victim",
    "victim_risk_assessment_made":                 "victim",
    "victim_criminal_history":                     "victim",
    "victim_history_abuse_as_victim":              "victim",
    "victim_history_abuse_as_offender":            "victim",
}

# Stage filter for predictive variant (P5)
PREDICTIVE_EXCLUDED_STAGES = {"during", "post"}

for col in FACTOR_AS_ATTR_COLS:
    raw = row.get(col)
    b   = parse_bool(raw)
    fkey = canonical_factor_key(col)
    stage = infer_stage(raw, col)

    # P5: skip during/post in predictive variant
    if GRAPH_VARIANT == "predictive" and stage in PREDICTIVE_EXCLUDED_STAGES:
        continue

    holder = RISK_HOLDER_MAP.get(col, "offender")
    holder_id = offender_id if holder == "offender" else victim_id
    set_factor_attrs(holder_id, fkey, b, stage)

# THREATENED_HARMED_CHILD: always set as rf_ attribute on offender (P9)
# AND create directed edges to child nodes when present (P2)
thc_bool = parse_bool(row.get("offender_threatened_harmed_children"))
fkey_thc = canonical_factor_key("offender_threatened_harmed_children")
set_factor_attrs(offender_id, fkey_thc, thc_bool, "prior")

if child_nodes:
    for ch_id in child_nodes:
        merge_edge_threestate(
            offender_id, ch_id, "THREATENED_HARMED_CHILD", thc_bool,
            scope="case", stage="prior",
            source_col="offender_threatened_harmed_children",
        )

print(f"Risk factors done. Nodes: {len(nodes)}, edges: {len(edges)}")

In [ ]:
# ── Cell 9: Artefacts — descriptive only (P3, P4) ──────────────────

if GRAPH_VARIANT == "descriptive":

    def _artefact_node(prefix, raw_val, attr_key):
        v = norm_str(raw_val)
        if v is None:
            return None
        sl = slugify(v)
        if sl is None:
            return None
        nid = f"{prefix}_{sl.upper()}"
        add_node(nid, prefix.lower(), label=v.title(), **{attr_key: v})
        return nid

    weapon_id   = _artefact_node("WEAPON",   row.get("weapon_type"),      "weapon_type")
    method_id   = _artefact_node("METHOD",   row.get("method_of_killing"), "method_type")
    location_id = _artefact_node("LOCATION", row.get("location_of_crime"), "location_type")

    if weapon_id:
        merge_edge(offender_id, weapon_id, "USED",    scope="case", stage="during")
        merge_edge(weapon_id,   victim_id, "AGAINST", scope="case", stage="during")

    if method_id:
        merge_edge(offender_id, method_id, "KILLED_BY", scope="case", stage="during")
        merge_edge(method_id,   victim_id, "SUFFERED",  scope="case", stage="during")

    if location_id:
        merge_edge(offender_id,  location_id, "LOCATED_AT", scope="case", stage="during")
        merge_edge(victim_id,    location_id, "LOCATED_AT", scope="case", stage="during")

    print(f"Artefacts: weapon={weapon_id}, method={method_id}, location={location_id}")
    print(f"Edges after artefacts: {len(edges)}")
else:
    print("Predictive variant — artefacts skipped (P4/P5).")

In [ ]:
# ── Cell 10: Court — descriptive only ──────────────────────────────

if GRAPH_VARIANT == "descriptive":
    court_name = norm_str(row.get("court_name"))
    court_num  = norm_str(row.get("court_number"))
    if court_name or court_num:
        slug = slugify(court_name or str(court_num))
        court_id = f"COURT_{slug.upper() if slug else 'UNKNOWN'}"
        add_node(court_id, "court", label=court_name or f"Court {court_num}",
                 court_name=court_name, court_number=court_num)
        merge_edge(offender_id, court_id, "SENTENCED_BY", scope="case", stage="post")
        merge_edge(case_id,     court_id, "SENTENCED_AT", scope="case", stage="post")
        print(f"Court node created: {court_id}")
    else:
        print("No court information available.")
else:
    print("Predictive variant — court skipped.")

In [ ]:
# ── Cell 11: Timeline events — descriptive only ─────────────────────

if GRAPH_VARIANT == "descriptive":

    def _parse_date(x):
        v = norm_str(x)
        if v is None:
            return None
        try:
            d = pd.to_datetime(v)
            return d
        except:
            return None

    crime_date   = _parse_date(row.get("crime_date"))
    report_date  = _parse_date(row.get("crime_report_date"))
    verdict_date = _parse_date(row.get("verdict_date"))

    event_ids = []

    if crime_date is not None:
        ev_id = f"EVENT_{cid}_CRIME"
        lbl = f"Crime ({crime_date.strftime('%H:%M') if crime_date.hour else 'date only'})"
        add_node(ev_id, "event", label=lbl, event_type="crime",
                 date=str(crime_date.date()),
                 day_of_week=crime_date.strftime("%A"),
                 month=crime_date.strftime("%B"),
                 stage="during")
        merge_edge(case_id, ev_id, "HAS_CRIME_EVENT", scope="case", stage="during")
        event_ids.append(ev_id)

    if report_date is not None:
        ev_id = f"EVENT_{cid}_REPORT"
        days = (report_date - crime_date).days if crime_date else None
        add_node(ev_id, "event", label="Reported to Authorities",
                 event_type="report",
                 date=str(report_date.date()),
                 days_from_crime=str(days) if days is not None else None,
                 stage="post")
        merge_edge(case_id, ev_id, "HAS_REPORT_EVENT", scope="case", stage="post")
        event_ids.append(ev_id)

    if verdict_date is not None:
        ev_id = f"EVENT_{cid}_VERDICT"
        days = (verdict_date - crime_date).days if crime_date else None
        add_node(ev_id, "event", label=f"Verdict ({days} days)" if days else "Verdict",
                 event_type="verdict",
                 date=str(verdict_date.date()),
                 days_from_crime=str(days) if days is not None else None,
                 stage="post")
        merge_edge(case_id, ev_id, "HAS_VERDICT_EVENT", scope="case", stage="post")
        event_ids.append(ev_id)

    # Chain events chronologically
    for i in range(len(event_ids) - 1):
        merge_edge(event_ids[i], event_ids[i+1], "FOLLOWED_BY", scope="case")

    print(f"Timeline events created: {event_ids}")
else:
    print("Predictive variant — timeline events skipped.")

In [ ]:
# ── Cell 12: Build DataFrames + Validation + Export ─────────────────

import xml.etree.ElementTree as ET

nodes_df = pd.DataFrame(list(nodes.values()))
edges_df = pd.DataFrame(list(edges.values()))

print("=== GRAPH SUMMARY ===")
print(f"Nodes: {len(nodes_df)}")
print(f"Edges: {len(edges_df)}")
print()
print("Node types:")
print(nodes_df["node_type"].value_counts().to_string())
print()
print("Edge types:")
print(edges_df["edge_type"].value_counts().to_string())
print()
print(f"Distinct edge types: {edges_df['edge_type'].nunique()}")

# ── Validation ────────────────────────────────────────────────────
print("\n=== VALIDATION ===")

# P1: no incident nodes
incident_nodes = nodes_df[nodes_df["node_type"] == "incident"]
if len(incident_nodes) == 0:
    print("✓ P1: No incident nodes")
else:
    print(f"⚠ P1 VIOLATION: {len(incident_nodes)} incident node(s) found")

# P4: no KILLED edge
killed_edges = edges_df[edges_df["edge_type"] == "KILLED"]
if len(killed_edges) == 0:
    print("✓ P4: No KILLED edge")
else:
    print(f"⚠ P4 VIOLATION: KILLED edge found")

# P5: in predictive, no during/post edges
if GRAPH_VARIANT == "predictive":
    if "stage" in edges_df.columns:
        leak_edges = edges_df[edges_df["stage"].isin(["during", "post"])]
        if len(leak_edges) == 0:
            print("✓ P5: No during/post edges in predictive variant")
        else:
            print(f"⚠ P5 VIOLATION: {len(leak_edges)} during/post edge(s):")
            print(leak_edges[["edge_type", "stage"]].to_string())
    else:
        print("ℹ P5: No 'stage' column in edges — cannot validate")

# P9: no factor nodes
factor_nodes = nodes_df[nodes_df["node_type"] == "factor"]
if len(factor_nodes) == 0:
    print("✓ P9: No factor nodes (rf_ attrs on person nodes)")
else:
    print(f"⚠ P9 VIOLATION: {len(factor_nodes)} factor node(s) found")

# P6: all dyadic edges have observed attribute
dyadic_types = (
    list(DYADIC_BEHAVIOR_MAP.values()) +
    list(DYADIC_SYMMETRIC.values()) +
    [s[1] for s in PRIOR_ACTION_SPECS] +
    ["SEPARATED_FROM", "HAS_NEW_PARTNER", "CHILD_CUSTODY_DISPUTE",
     "SHARED_CHILDREN", "PRIOR_FAMILY_COURT_INVOLVEMENT"]
)
if "observed" in edges_df.columns:
    dyadic_df = edges_df[edges_df["edge_type"].isin(dyadic_types)]
    missing_obs = dyadic_df[dyadic_df["observed"].isna()]
    if len(missing_obs) == 0:
        print(f"✓ P6: All {len(dyadic_df)} dyadic edges have observed attribute")
    else:
        print(f"⚠ P6: {len(missing_obs)} dyadic edge(s) missing observed")

# ── Build NetworkX MultiDiGraph ──────────────────────────────────
G = nx.MultiDiGraph()
for _, n in nodes_df.iterrows():
    G.add_node(n["node_id"], **{k: v for k, v in n.items() if pd.notna(v)})
for _, e in edges_df.iterrows():
    attrs = {k: v for k, v in e.items()
             if k not in {"source_id", "target_id"} and pd.notna(v)}
    G.add_edge(e["source_id"], e["target_id"], weight=1.0, **attrs)

print(f"\nNetworkX graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

# ── Export ──────────────────────────────────────────────────────
nodes_csv = EXPORT_DIR / f"case_{cid}_nodes.csv"
edges_csv = EXPORT_DIR / f"case_{cid}_edges.csv"
triples_tsv = EXPORT_DIR / f"case_{cid}_triples.tsv"
gexf_path   = EXPORT_DIR / f"case_{cid}.gexf"

nodes_df.to_csv(nodes_csv, index=False)
edges_df.to_csv(edges_csv, index=False)

# TSV triples: source \t edge_type \t target \t attrs_json
with open(triples_tsv, "w") as f:
    f.write("source\tedge_type\ttarget\tattrs\n")
    for _, e in edges_df.iterrows():
        attrs = {k: v for k, v in e.items()
                 if k not in {"edge_id", "source_id", "target_id", "edge_type", "edge_key"}
                 and pd.notna(v)}
        f.write(f"{e['source_id']}\t{e['edge_type']}\t{e['target_id']}\t{json.dumps(attrs)}\n")

nx.write_gexf(G, gexf_path)

# Fix Gephi big-arrow: add viz:thickness=1.0 to all edges
tree = ET.parse(gexf_path)
root = tree.getroot()
ns = {"gexf": "http://gexf.net/1.3", "viz": "http://gexf.net/1.3/viz"}
for edge_el in root.iter("{http://gexf.net/1.3}edge"):
    thick = ET.SubElement(edge_el, "{http://gexf.net/1.3/viz}thickness")
    thick.set("value", "1.0")
tree.write(gexf_path, xml_declaration=True, encoding="utf-8")

print(f"\nExported to {EXPORT_DIR}/")
print(f"  nodes.csv, edges.csv, triples.tsv, .gexf")

## Section 6 — GNN Prototype: HeteroData + Forward Pass

Demonstrates that the v7 graph is compatible with a heterogeneous GNN.

**Pipeline:**
1. Convert `nodes`/`edges` dicts → `HeteroData`
2. Build model with `to_hetero(SAGEConv)` — one conv function per edge-type
3. Run forward pass → offender embedding + risk score

This is an **architecture prototype** on one case.
Actual training and evaluation requires a larger dataset.

> **Requirement:** Run notebook with `GRAPH_VARIANT = "predictive"` before this section.

In [ ]:
# ── GNN Cell 1: Imports + assertion ────────────────────────────────
import torch
import torch.nn.functional as F
from torch_geometric.data import HeteroData
from torch_geometric.nn import SAGEConv, to_hetero
import torch_geometric.transforms as T
from collections import defaultdict

assert GRAPH_VARIANT == "predictive", (
    "Set GRAPH_VARIANT = 'predictive' and re-run the notebook. "
    "The GNN prototype uses only prior-stage data (P5)."
)
print("✓ GRAPH_VARIANT = predictive")

In [ ]:
# ── GNN Cell 2: Node-index mapping ─────────────────────────────────
# HeteroData requires integer indices per node-type.
# We split 'person' nodes by role: offender, victim, child.

node_type_map = {}   # node_id → (type_str, int_index)

node_type_map[offender_id] = ("offender", 0)
node_type_map[victim_id]   = ("victim",   0)
node_type_map[case_id]     = ("case",     0)
for i, ch_id in enumerate(child_nodes):
    node_type_map[ch_id] = ("child", i)

print("Node-type mapping:")
for nid, (ntype, idx) in node_type_map.items():
    print(f"  {nid[:50]:<50} → {ntype}[{idx}]")

In [ ]:
# ── GNN Cell 3: Feature vectors from rf_* attributes ───────────────

def rf_feature_vector(node_dict):
    """
    Extract rf_*_value and rf_*_observed from a node dict and return
    a float tensor. Three-state: value ∈ {0, 0.5, 1}, observed ∈ {0, 1}.
    Sort keys for determinism (P8).
    """
    rf_keys = sorted([k for k in node_dict if k.endswith("_value") and k.startswith("rf_")])
    vec = []
    for k in rf_keys:
        base = k[:-6]  # strip "_value"
        vec.append(float(node_dict.get(k, 0.5)))
        vec.append(float(node_dict.get(f"{base}_observed", 0)))
    return vec


def age_feature(node_dict, max_age=100.0):
    """Normalised age feature [0,1]. Returns 0.5 if missing."""
    age_raw = node_dict.get("age")
    if age_raw is None:
        return [0.5]
    try:
        return [float(str(age_raw).split()[0]) / max_age]
    except:
        return [0.5]


def node_feature_vector(node_dict):
    """Concatenate age + rf_* features."""
    return age_feature(node_dict) + rf_feature_vector(node_dict)


# Build feature tensors for each node type
offender_feat = node_feature_vector(nodes[offender_id])
victim_feat   = node_feature_vector(nodes[victim_id])
case_feat     = [1.0]  # constant placeholder — no meaningful numeric attrs
child_feats   = [[1.0] for _ in child_nodes]  # constant placeholder

print(f"Offender feature vector length: {len(offender_feat)}")
print(f"Victim feature vector length:   {len(victim_feat)}")
print(f"Number of rf_*_value keys (offender): {len([k for k in nodes[offender_id] if k.endswith('_value')])}")
print(f"Number of rf_*_value keys (victim):   {len([k for k in nodes[victim_id]   if k.endswith('_value')])}")

In [ ]:
# ── GNN Cell 4: Build HeteroData ───────────────────────────────────

data = HeteroData()

# Node features
data["offender"].x = torch.tensor([offender_feat], dtype=torch.float)
data["victim"].x   = torch.tensor([victim_feat],   dtype=torch.float)
data["case"].x     = torch.tensor([case_feat],     dtype=torch.float)
if child_nodes:
    data["child"].x = torch.tensor(child_feats, dtype=torch.float)

# Build edge_index per (src_type, edge_type, dst_type)
edge_lists = defaultdict(lambda: ([], []))  # (src_type, rel, dst_type) → ([src_idx], [dst_idx])

for e in edges.values():
    src_id = e["source_id"]
    dst_id = e["target_id"]
    if src_id not in node_type_map or dst_id not in node_type_map:
        continue
    src_type, src_idx = node_type_map[src_id]
    dst_type, dst_idx = node_type_map[dst_id]
    rel = e["edge_type"].lower()
    edge_lists[(src_type, rel, dst_type)][0].append(src_idx)
    edge_lists[(src_type, rel, dst_type)][1].append(dst_idx)

for (src_type, rel, dst_type), (srcs, dsts) in edge_lists.items():
    data[(src_type, rel, dst_type)].edge_index = torch.tensor(
        [srcs, dsts], dtype=torch.long
    )

# Add reverse edges for message passing (ToUndirected)
data = T.ToUndirected()(data)

print("HeteroData object:")
print(data)
print(f"\nNode types: {data.node_types}")
print(f"Edge types ({len(data.edge_types)}):")
for et in data.edge_types:
    print(f"  {et}")

In [ ]:
# ── GNN Cell 5: Model definition + forward pass ─────────────────────

HIDDEN_DIM = 64


class GNN(torch.nn.Module):
    """Two-layer GraphSAGE base. to_hetero() expands to one copy per edge-type."""
    def __init__(self, hidden_dim):
        super().__init__()
        self.conv1 = SAGEConv((-1, -1), hidden_dim)
        self.conv2 = SAGEConv((-1, -1), hidden_dim)

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = self.conv2(x, edge_index)
        return x


class RiskClassifier(torch.nn.Module):
    """GNN embedding → risk score per offender node. Output ∈ (0,1) via sigmoid."""
    def __init__(self, hidden_dim):
        super().__init__()
        self.gnn        = to_hetero(GNN(hidden_dim), data.metadata(), aggr="sum")
        self.classifier = torch.nn.Linear(hidden_dim, 1)

    def forward(self, x_dict, edge_index_dict):
        embeddings = self.gnn(x_dict, edge_index_dict)
        offender_emb = embeddings["offender"]          # [n_offenders, hidden_dim]
        score = torch.sigmoid(self.classifier(offender_emb))
        return score, embeddings


model = RiskClassifier(hidden_dim=HIDDEN_DIM)
model.eval()

with torch.no_grad():
    risk_score, all_embeddings = model(data.x_dict, data.edge_index_dict)

print("=" * 55)
print("GNN forward pass — output")
print("=" * 55)
print(f"\nModel parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"\nOffender risk score : {risk_score.item():.4f}  (untrained — random init)")
print(f"Offender embedding  : shape {all_embeddings['offender'].shape}")
print(f"Victim embedding    : shape {all_embeddings['victim'].shape}")
print(f"Case embedding      : shape {all_embeddings['case'].shape}")
print(f"\n✓ Forward pass succeeded")
print(f"  Input:  HeteroData with {len(data.metadata()[0])} node types, "
      f"{len(data.metadata()[1])} edge types")
print(f"  Output: risk score ∈ (0,1) per offender node")
print(f"\n  Architecture scales to N cases when the dataset grows:")
print(f"  offender.x → [N, {data['offender'].x.shape[1]}]")
print(f"  victim.x   → [N, {data['victim'].x.shape[1]}]")